# MOESCORE XACLE 2027 — Full8 end-to-end on Kaggle T4 ×2

This notebook builds independent profile `full_8`: original E1–E7 order plus the frozen M2D-CLAP audio/text LoRA CORAL Expert 8. It verifies and reuses a compatible Full7 store when available. Otherwise, it runs the existing Full9 → E5 → Whisper → E6 → Full7 feature-store stages. The Full7 Gate is never trained.

Attach the official XACLE dataset, the Full9 cache (unless you attach a validated Full7 store), and pretrained weights containing `m2d-lora-Nam.pt` plus the exact M2D `checkpoint-30.pth`. Exact-name discovery runs under `/kaggle/input`; optional path overrides are in the central configuration. Both checkpoint identities are verified before any E5 or Whisper work.

`ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP=True` is an explicit research exception. Validation can be optimistically biased. `SKIP_SCORE_REVIEW_EXPERIMENTAL=True` skips human review without claiming or fabricating it; the train-only Score Audit remains on. Test labels are opened only after label-free predictions are frozen.


## 00 — Central configuration

Set paths or switches here. Leave both Expert 8 checkpoint overrides as `None` when exactly one matching artifact is attached in Kaggle Inputs. `Save & Run All` performs the stages in order without a Human Score Review pause.


In [ ]:
from pathlib import Path
import gc
import os, sys, json, csv, subprocess, shutil, hashlib, tarfile, tempfile, time

REPO_URL = "https://github.com/vietquang8071/MOESCORE.git"
SOURCE_COMMIT = "ffcb969951731dccca792ee86b6bdadb088b2f3c"
REPO_ROOT = Path("/kaggle/working/MOESCORE")
SKIP_SCORE_REVIEW_EXPERIMENTAL = True
RUN_TEST_METRICS = True
DATA_ROOT = Path("/kaggle/input/datasets/qnv8071/xacle-2027-data/XACLE_Challenge_2027_development_dataset")
FULL9_CACHE_INPUT = Path("/kaggle/input/datasets/qnv8071/moescore-full9-feature-cache")
FULL9_STORE_OVERRIDE = None

WORK_ROOT = Path("/kaggle/working/moescore_full8")
CONFIG_ROOT = WORK_ROOT / "configs"
LOG_ROOT = WORK_ROOT / "logs"
WHISPER_ROOT = WORK_ROOT / "whisper_cache"
WHISPER_ARCHIVE = WORK_ROOT / "whisper_feature_cache.tar"
WHISPER_ARCHIVE_MANIFEST = WORK_ROOT / "whisper_feature_cache_archive.json"
E5_ROOT_BASE = WORK_ROOT / "e5"
E5_ROOT = E5_ROOT_BASE
E6_ROOT_BASE = WORK_ROOT / "e6"
E6_ROOT = E6_ROOT_BASE
FULL7_ROOT = WORK_ROOT / "full7_store"
E8_CACHE_ROOT = WORK_ROOT / "expert8_cache"
FULL8_ROOT = WORK_ROOT / "full8_store"
GATE_ROOT_BASE = WORK_ROOT / "full8_gate"
GATE_ROOT = GATE_ROOT_BASE
METADATA_ROOT = WORK_ROOT / "whisper_metadata"
HF_E8_CACHE_ROOT = WORK_ROOT / "hf_e8_cache"
FULL7_STORE_OVERRIDE = None
E8_LORA_CHECKPOINT_OVERRIDE = None
E8_BASE_CHECKPOINT_OVERRIDE = None
SCORE_REVIEW_FILE_OVERRIDE = None

WHISPER_SNAPSHOT_INPUT = None
WHISPER_CACHE_DIR = WORK_ROOT / "hf_whisper" / "hub"
ALLOW_WHISPER_DOWNLOAD = True
WHISPER_BATCH_SIZE = 1
WHISPER_NUM_WORKERS = 0

INSTALL_MISSING_DEPENDENCIES = True
ALLOW_E8_BERT_DOWNLOAD = True
E8_BATCH_SIZE = 1
E8_SHARD_SIZE = 8
RUN_E5_TRAIN = True
RUN_WHISPER_SMOKE = True
RUN_WHISPER_EXTRACT = True
RUN_E6_TRAIN = True
RUN_FULL7_BUILD = True
RUN_E8_EXTRACTION = True
RUN_FULL8_BUILD = True
RUN_SCORE_AUDIT = True
RUN_GATE_TRAIN = True
RUN_VALIDATION = True
RUN_TEST_INFERENCE = True
ALLOW_POST_INFERENCE_TEST_LABEL_ACCESS = True
ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP = True

HUMAN_SCORE_REVIEW_CONFIRMED = False
REVIEWER_NAME = ""
REVIEW_EVIDENCE = ""
RUN_PURPOSE = "research_only"

E5_DEVICE = "0"
E6_DEVICE = "0"
GATE_DEVICE = "cuda:0"
SHARD_SIZE = 32
EXPECTED_COUNTS = {"train": 7500, "validation": 3000, "test": 3000}
FULL8_FEATURE_WIDTH = 33032
FULL8_SCORE_WIDTH = 8

CREATE_LOCAL_BACKUP = True
CREATE_PRE_GATE_BACKUP = True
UPLOAD_PRIVATE_PRE_GATE_BACKUP = False
UPLOAD_PRIVATE_BACKUP = False
PRIVATE_BACKUP_HANDLE = "qnv8071/moescore-full7-intermediates"

os.environ["HF_HOME"] = str(HF_E8_CACHE_ROOT)
os.environ["HF_HUB_CACHE"] = str(HF_E8_CACHE_ROOT / "hub")


def atomic_write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_name = tempfile.mkstemp(
        prefix=path.name + ".", suffix=".tmp", dir=path.parent,
    )
    try:
        with os.fdopen(descriptor, "w", encoding="utf-8") as stream:
            json.dump(value, stream, indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False)
            stream.write("\n")
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary_name, path)
    finally:
        Path(temporary_name).unlink(missing_ok=True)


assert WHISPER_BATCH_SIZE == 1 and E8_BATCH_SIZE == 1
for directory in (
    WORK_ROOT, CONFIG_ROOT, LOG_ROOT, WHISPER_ROOT, E5_ROOT_BASE, E6_ROOT_BASE,
    FULL7_ROOT, E8_CACHE_ROOT, FULL8_ROOT, GATE_ROOT_BASE, METADATA_ROOT,
    WHISPER_CACHE_DIR, HF_E8_CACHE_ROOT,
):
    directory.mkdir(parents=True, exist_ok=True)
print("Full8 work root:", WORK_ROOT)


## 01 — Source, E8 checkpoint and two-T4 preflight

The exact `m2d-lora-Nam.pt` list checkpoint and compatible M2D base checkpoint are validated before E5 or Whisper extraction. Expert 8 remains independent from the existing E7 M2D-LoRA adapter.


In [ ]:
if not REPO_ROOT.is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--detach", SOURCE_COMMIT], check=True)
current_sha = subprocess.check_output(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True,
).strip()
if current_sha != SOURCE_COMMIT:
    raise ValueError(f"Repository checkout differs from the pinned source: {current_sha}")

required_files = [
    "scripts/build_full7_feature_store.py", "scripts/audit_full7_scores.py",
    "scripts/train_set_a_residual.py", "scripts/train_set_b_residual.py",
    "scripts/extract_whisper_features.py", "scripts/train_moe_gating.py",
    "scripts/evaluate_moe_gating.py", "scripts/infer_moe_gating.py",
    "scripts/install_kaggle_dependencies.py", "configs/moe_full7.example.yaml",
    "configs/moe_full8.example.yaml", "configs/expert8_extraction.example.yaml",
    "scripts/extract_expert8_features.py", "scripts/build_full8_feature_store.py",
    "data/full8_assembler.py", "model/expert_adapters/expert8_m2d_clap_adapter.py",
]
missing_files = [name for name in required_files if not (REPO_ROOT / name).is_file()]
if missing_files:
    raise FileNotFoundError(f"Pinned Full8 source is incomplete: {missing_files}")

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Source checkout ready without importing the PyTorch/CUDA stack:", current_sha)


## 02 — Kaggle dependencies (reuse the repository's Python-specific installer)

Keep Kaggle's preinstalled torch/torchaudio/torchvision stack. This follows the exact dependency installation strategy used in the latest Full9 notebook. A kernel restart may be necessary if core libraries already imported are replaced. Turn `INSTALL_MISSING_DEPENDENCIES=False` only after a verified installation.

In [ ]:
import importlib.metadata as package_metadata

GPU_STACK_PACKAGES = {"torch", "torchaudio", "torchvision", "triton"}
preloaded_gpu_modules = [
    name for name in ("torch", "torchaudio", "torchvision", "triton")
    if name in sys.modules
]
preloaded_gpu_modules.extend(sorted(
    name for name in sys.modules if name.startswith("nvidia.")
))
if INSTALL_MISSING_DEPENDENCIES and preloaded_gpu_modules:
    raise RuntimeError(
        "torch already imported or CUDA modules loaded before dependency installation: "
        f"{preloaded_gpu_modules}. Restart the Kaggle kernel and use Save & Run All."
    )

dependency_report_path = LOG_ROOT / "dependencies_install.json"
if INSTALL_MISSING_DEPENDENCIES:
    if sys.version_info[:2] == (3, 12):
        requirements_file = REPO_ROOT / "requirements-kaggle-py312.txt"
        legacy_no_deps = False
    elif sys.version_info[:2] == (3, 13):
        requirements_file = REPO_ROOT / "requirements-kaggle-py313.txt"
        legacy_no_deps = True
    else:
        raise RuntimeError(f"Kaggle execution requires Python 3.12 or 3.13; got {sys.version_info[:2]}")
    install_command = [
        sys.executable, str(REPO_ROOT / "scripts/install_kaggle_dependencies.py"),
        "--requirements", str(requirements_file),
        "--constraints", str(WORK_ROOT / "torch_constraints.txt"),
        "--report-json", str(dependency_report_path),
    ]
    print("Installing Python dependencies with the Kaggle GPU stack pinned before model imports.")
    subprocess.run(install_command, cwd=REPO_ROOT, check=True)
    if legacy_no_deps:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--no-deps", "msclap==1.3.4", "laion-clap==1.1.7"],
            check=True,
        )
    install_report = json.loads(dependency_report_path.read_text(encoding="utf-8"))
    if install_report.get("status") != "PASS":
        raise RuntimeError("Kaggle dependency installer did not report PASS")
    frozen_stack = install_report.get("gpu_stack_preserved")
    if not isinstance(frozen_stack, dict) or not {"torch", "torchaudio", "torchvision"}.issubset(frozen_stack):
        raise RuntimeError("Dependency report lacks the preinstalled torch/torchaudio/torchvision identities")
    changed = install_report.get("changed_distributions", {})
    unsafe_changes = [
        name for name in changed
        if name in frozen_stack or name in GPU_STACK_PACKAGES or name.startswith("nvidia-")
    ]
    if unsafe_changes:
        raise RuntimeError(f"Dependency installation changed the active CUDA stack: {unsafe_changes}")
    for name, expected_version in frozen_stack.items():
        if package_metadata.version(name) != expected_version:
            raise RuntimeError(f"Installed GPU dependency {name} changed after the verified install")
else:
    frozen_stack = {
        name: package_metadata.version(name)
        for name in GPU_STACK_PACKAGES
        if any(dist.metadata.get("Name", "").lower().replace("_", "-") == name
               for dist in package_metadata.distributions())
    }

import torch
import numpy as np
import torchaudio
import torchvision

for package_name, imported_version in (
    ("torch", torch.__version__), ("torchaudio", torchaudio.__version__),
    ("torchvision", torchvision.__version__),
):
    if package_metadata.version(package_name) != imported_version:
        raise RuntimeError(f"Imported {package_name} does not match its installed package metadata")
    if package_name in frozen_stack and imported_version != frozen_stack[package_name]:
        raise RuntimeError(f"Imported {package_name} differs from the pre-install CUDA stack identity")
if torch.version.cuda is None or not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError("Select a Kaggle GPU T4 ×2 runtime with a working CUDA-enabled PyTorch stack")
gpu_names = [torch.cuda.get_device_name(index) for index in range(2)]
if any("T4" not in name.upper() for name in gpu_names):
    raise RuntimeError(f"Expected two T4 GPUs; found {gpu_names}")
print("Verified Kaggle GPU stack:", frozen_stack)
for index, name in enumerate(gpu_names):
    props = torch.cuda.get_device_properties(index)
    print(f"cuda:{index}: {name}, {props.total_memory / 2**30:.2f} GiB")

from model.expert_adapters.expert8_m2d_clap_runtime import (
    EXPECTED_BASE_M2D_SHA256, EXPECTED_LORA_SHAPES,
    load_expert8_checkpoint, validate_base_m2d_checkpoint,
)
from data.moe_expert_cache import sha256_file

def resolve_exact_input(filename, override, local_fallback=None):
    if override is not None:
        candidate = Path(override).expanduser().resolve()
        if not candidate.is_file():
            raise FileNotFoundError(f"Checkpoint override not found: {candidate}")
        return candidate
    input_root = Path("/kaggle/input")
    matches = sorted(
        candidate.resolve() for candidate in input_root.rglob(filename) if candidate.is_file()
    ) if input_root.is_dir() else []
    if len(matches) > 1:
        raise ValueError(
            f"Ambiguous {filename}: found {len(matches)} exact matches: {matches}. "
            "Set the corresponding explicit override."
        )
    if len(matches) == 1:
        return matches[0]
    if local_fallback is not None and Path(local_fallback).is_file():
        return Path(local_fallback).resolve()
    raise FileNotFoundError(
        f"Required {filename} is missing under /kaggle/input and {local_fallback}; "
        "attach the checkpoint Dataset or set an override."
    )

E8_LORA_CHECKPOINT = resolve_exact_input(
    "m2d-lora-Nam.pt", E8_LORA_CHECKPOINT_OVERRIDE,
    REPO_ROOT / "weights" / "m2d-lora-Nam.pt",
)
E8_BASE_CHECKPOINT = resolve_exact_input(
    "checkpoint-30.pth", E8_BASE_CHECKPOINT_OVERRIDE,
    REPO_ROOT / "weights" / "pretrained" / "checkpoint-30.pth",
)
E8_LORA_SHA256 = sha256_file(E8_LORA_CHECKPOINT)
E8_BASE_SHA256 = validate_base_m2d_checkpoint(E8_BASE_CHECKPOINT)
e8_preflight = load_expert8_checkpoint(E8_LORA_CHECKPOINT)
if len(e8_preflight["lora"]) != len(EXPECTED_LORA_SHAPES) or len(e8_preflight["lora"]) != 24:
    raise ValueError("E8 preflight requires exactly 24 ordered LoRA tensors")
if tuple(e8_preflight["head"]["net.0.weight"].shape) != (1024, 3073):
    raise ValueError("E8 CORAL head input dimensions are incompatible")
if tuple(e8_preflight["head"]["net.5.weight"].shape) != (10, 256):
    raise ValueError("E8 CORAL head must output exactly ten CORAL logits")
print("E8 trained checkpoint:", E8_LORA_CHECKPOINT, E8_LORA_SHA256)
print("E8 base checkpoint:", E8_BASE_CHECKPOINT, E8_BASE_SHA256)
print("E8 keys, 3073-to-10 CORAL head and all 24 LoRA tensors: PASS")
del e8_preflight

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"Attach the official XACLE dataset: {DATA_ROOT}")
for split, expected_count in EXPECTED_COUNTS.items():
    if not (DATA_ROOT / "wav" / split).is_dir():
        raise FileNotFoundError(f"Missing XACLE {split} WAV directory")
    if not (DATA_ROOT / "meta_data" / f"{split}_average.csv").is_file():
        raise FileNotFoundError(f"Missing XACLE {split} metadata CSV")
    observed = sum(1 for path in (DATA_ROOT / "wav" / split).glob("*.wav") if path.is_file())
    if observed != expected_count:
        raise ValueError(f"{split}: expected {expected_count} WAV files, found {observed}")
print("XACLE sample counts:", EXPECTED_COUNTS)


In [ ]:
from data.moe_feature_store import validate_feature_store
from data.audio_overlap import build_train_validation_audio_overlap_diagnostic

def verify_full7_candidate(root):
    root = Path(root).expanduser().resolve(strict=True)
    manifests = {}
    for split, expected in EXPECTED_COUNTS.items():
        manifest = validate_feature_store(root, split=split, require_labels=split != "test")
        count = sum(shard["count"] for shard in manifest["shards"])
        if (manifest.get("expert_profile"), manifest.get("feature_width"),
                manifest.get("score_width"), count) != ("full_7", 29959, 7, expected):
            raise ValueError(f"{split}: incompatible Full7 profile, dimensions or sample count")
        if manifest.get("labels_present") is not (split != "test"):
            raise ValueError(f"{split}: invalid Full7 label presence")
        rows = manifest["assembly_identity"]["ordered_source_fingerprints"]
        shard_ids = [sample_id for shard in manifest["shards"] for sample_id in shard["sample_ids"]]
        shard_names = [name for shard in manifest["shards"] for name in shard["wave_file_names"]]
        if (len(rows) != expected
                or [row.get("sample_id") for row in rows] != shard_ids
                or [row.get("wave_file_name") for row in rows] != shard_names):
            raise ValueError(f"{split}: Full7 per-split source membership/alignment mismatch")
        manifests[split] = manifest

    # Source manifest identities are split-scoped; each split has already been
    # validated against its own signed manifest and exact ordered membership.
    train_rows = manifests["train"]["assembly_identity"]["ordered_source_fingerprints"]
    validation_rows = manifests["validation"]["assembly_identity"]["ordered_source_fingerprints"]
    test_rows = manifests["test"]["assembly_identity"]["ordered_source_fingerprints"]
    test_hashes = {row["wav_sha256"] for row in train_rows + validation_rows}.intersection(
        row["wav_sha256"] for row in test_rows
    )
    if test_hashes:
        raise ValueError(f"Full7 duplicate WAV involves official test: {sorted(test_hashes)[0]}")
    diagnostic = build_train_validation_audio_overlap_diagnostic(
        train_rows, validation_rows,
        allow_train_validation_audio_overlap=ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP,
    )
    if diagnostic["overlapping_sha256_count"] and not ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP:
        from data.audio_overlap import raise_for_audio_overlap
        raise_for_audio_overlap(diagnostic, stage="Full7 cache verification")
    return manifests, diagnostic


if FULL7_STORE_OVERRIDE is not None:
    FULL7_ROOT = Path(FULL7_STORE_OVERRIDE).expanduser().resolve()
    full7_manifests, full7_overlap_diagnostic = verify_full7_candidate(FULL7_ROOT)
    NEED_FULL7_BUILD = False
else:
    candidates = []
    if FULL7_ROOT.is_dir():
        candidates.append(FULL7_ROOT.resolve())
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for manifest_path in input_root.rglob("train/manifest.json"):
            candidate = manifest_path.parent.parent.resolve()
            if candidate not in candidates:
                candidates.append(candidate)
    valid, invalid = [], []
    for candidate in candidates:
        try:
            valid.append((candidate, verify_full7_candidate(candidate)))
        except Exception as exc:
            invalid.append((candidate, f"{type(exc).__name__}: {exc}"))
    if len(valid) > 1:
        raise ValueError(f"Multiple compatible Full7 stores found; set FULL7_STORE_OVERRIDE: {[str(item[0]) for item in valid]}")
    if valid:
        FULL7_ROOT = valid[0][0]
        full7_manifests, full7_overlap_diagnostic = valid[0][1]
        NEED_FULL7_BUILD = False
        print("Verified and reusing Full7 store:", FULL7_ROOT)
        print("Train/validation overlap:", json.dumps(full7_overlap_diagnostic, sort_keys=True))
    else:
        FULL7_ROOT = WORK_ROOT / "full7_store"
        NEED_FULL7_BUILD = True
        print("No compatible Full7 store found; existing Full9/E5/Whisper/E6 stages will run.")
        for candidate, error in invalid[:10]:
            print("Ignored incompatible Full7 candidate:", candidate, error)
print("NEED_FULL7_BUILD =", NEED_FULL7_BUILD)


## 03 — E8 BERT resource preflight

Establish the exact `google-bert/bert-base-uncased` snapshot identity before E5 or Whisper work. If absent, fetch it now when enabled; otherwise fail before expensive stages.


In [ ]:
from scripts.extract_moe_feature_store import _text_snapshot_identity


def prepare_e8_bert_snapshot(
    cache_root, *, allow_download, snapshot_download_fn=None, snapshot_identity_fn=None,
):
    if type(allow_download) is not bool:
        raise TypeError("allow_download must be a boolean")
    if snapshot_download_fn is None:
        from huggingface_hub import snapshot_download as snapshot_download_fn
    if snapshot_identity_fn is None:
        snapshot_identity_fn = _text_snapshot_identity

    cache_root = Path(cache_root).expanduser().resolve()
    hub_root = cache_root / "hub"
    hub_root.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(cache_root)
    os.environ["HF_HUB_CACHE"] = str(hub_root)
    allow_patterns = [
        "config.json", "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
        "vocab.txt", "added_tokens.json", "model.safetensors", "model.safetensors.index.json",
        "model-*.safetensors", "pytorch_model.bin", "pytorch_model.bin.index.json",
        "pytorch_model-*.bin",
    ]

    def verify_snapshot_assets(snapshot_path):
        snapshot_path = Path(snapshot_path).expanduser().resolve(strict=True)
        config_path = snapshot_path / "config.json"
        tokenizer_config = snapshot_path / "tokenizer_config.json"
        if not config_path.is_file() or not tokenizer_config.is_file():
            raise FileNotFoundError("E8 BERT snapshot requires config.json and tokenizer_config.json")
        config = json.loads(config_path.read_text(encoding="utf-8"))
        if config.get("model_type") != "bert" or config.get("hidden_size") != 768:
            raise ValueError("E8 requires the 768-dimensional BERT base snapshot")
        if not ((snapshot_path / "vocab.txt").is_file() or (snapshot_path / "tokenizer.json").is_file()):
            raise FileNotFoundError("E8 BERT snapshot is missing tokenizer vocab assets")

        weights = [snapshot_path / name for name in ("model.safetensors", "pytorch_model.bin")]
        has_weights = any(path.is_file() and path.stat().st_size > 0 for path in weights)
        for index_name in ("model.safetensors.index.json", "pytorch_model.bin.index.json"):
            index_path = snapshot_path / index_name
            if not index_path.is_file():
                continue
            index = json.loads(index_path.read_text(encoding="utf-8"))
            shard_names = sorted(set(index.get("weight_map", {}).values()))
            if not shard_names:
                raise ValueError(f"E8 BERT weight index is empty: {index_path}")
            for shard_name in shard_names:
                shard_path = (snapshot_path / shard_name).resolve(strict=True)
                if (not shard_path.is_relative_to(snapshot_path)
                        or not shard_path.is_file() or shard_path.stat().st_size <= 0):
                    raise FileNotFoundError(f"E8 BERT model shard is missing, empty or escapes the snapshot: {shard_name}")
            has_weights = True
        if not has_weights:
            raise FileNotFoundError("E8 BERT snapshot is missing PyTorch model weights")
        return snapshot_path

    try:
        snapshot_path = verify_snapshot_assets(snapshot_download_fn(
            "google-bert/bert-base-uncased", cache_dir=str(hub_root),
            local_files_only=True, allow_patterns=allow_patterns,
        ))
    except Exception as cache_error:
        if not allow_download:
            raise FileNotFoundError(
                "Complete google-bert/bert-base-uncased PyTorch/tokenizer assets are absent "
                "from HF_E8_CACHE_ROOT/hub; attach the cache or enable ALLOW_E8_BERT_DOWNLOAD."
            ) from cache_error
        print("Fetching only the required E8 BERT config, PyTorch weights and tokenizer files.")
        snapshot_path = verify_snapshot_assets(snapshot_download_fn(
            "google-bert/bert-base-uncased", cache_dir=str(hub_root),
            local_files_only=False, allow_patterns=allow_patterns,
        ))

    identity = snapshot_identity_fn(
        "google-bert/bert-base-uncased", {"cache_dir": str(hub_root)}, {},
    )
    revision = snapshot_path.name
    snapshots = [
        snapshot for view in identity.get("cache_views", [])
        for snapshot in view.get("snapshots", [])
    ]
    if not any(snapshot.get("revision") == revision for snapshot in snapshots):
        raise ValueError("E8 BERT snapshot identity did not include the validated snapshot revision")
    return snapshot_path, identity


e8_bert_snapshot, e8_bert_identity = prepare_e8_bert_snapshot(
    HF_E8_CACHE_ROOT, allow_download=ALLOW_E8_BERT_DOWNLOAD,
)
print("E8 BERT snapshot:", e8_bert_snapshot)
print("E8 BERT cache identity verified with _text_snapshot_identity")
del e8_bert_identity


## 04 — Resolve and verify the existing Full9 store

This notebook **never** reruns nine expert backbones. The Full9 Kaggle Input may contain either a direct store or the archived `feature_cache.tar` format produced by `scripts/kaggle_full9_artifacts.py`. Archive restore is hash-verified. Source manifests are mandatory because the downstream E5/E6/Core builder needs exact WAV/caption identities.

In [ ]:
if NEED_FULL7_BUILD:
    from scripts.kaggle_full9_artifacts import restore_feature_cache, validate_full9_store
    from data.moe_feature_store import validate_feature_store
    from data.set_a_cached_dataset import _read_source_manifest

    if not DATA_ROOT.is_dir():
        raise FileNotFoundError(f"Missing official XACLE input: {DATA_ROOT}")
    for split in EXPECTED_COUNTS:
        assert (DATA_ROOT / "wav" / split).is_dir(), f"Missing audio directory: {split}"

    if FULL9_STORE_OVERRIDE is not None:
        FULL9_ROOT = Path(FULL9_STORE_OVERRIDE).expanduser().resolve()
    else:
        if not FULL9_CACHE_INPUT.exists():
            raise FileNotFoundError(f"Attach your existing Full9 cache Dataset: {FULL9_CACHE_INPUT}. Available: {list(Path('/kaggle/input').iterdir())}")
        roots = []
        for candidate_manifest in FULL9_CACHE_INPUT.rglob("train/manifest.json"):
            candidate = candidate_manifest.parent.parent
            if all((candidate / split / "manifest.json").is_file() for split in EXPECTED_COUNTS):
                try:
                    body = json.loads(candidate_manifest.read_text())
                    if body.get("expert_profile") == "full_9":
                        roots.append(candidate)
                except (OSError, ValueError):
                    pass
        if len(roots) == 1:
            FULL9_ROOT = roots[0]
        elif len(roots) > 1:
            raise ValueError(f"Multiple Full9 store roots; set FULL9_STORE_OVERRIDE explicitly: {roots}")
        else:
            # The Kaggle upload format contains feature_cache.tar + upload_manifest.json.
            archives = list(FULL9_CACHE_INPUT.rglob("upload_manifest.json"))
            if len(archives) != 1:
                raise FileNotFoundError("No direct Full9 store; expected exactly one upload_manifest.json + feature_cache.tar. Set FULL9_STORE_OVERRIDE if needed.")
            destination = WORK_ROOT / "full9_restore"
            archive_dir = archives[0].parent
            if not (destination / "feature_cache").is_dir():
                restore_feature_cache(archive_dir, destination)
            roots = []
            for manifest_path in (destination / "feature_cache").rglob("train/manifest.json"):
                candidate = manifest_path.parent.parent
                if all((candidate / split / "manifest.json").is_file() for split in EXPECTED_COUNTS):
                    if json.loads(manifest_path.read_text()).get("expert_profile") == "full_9":
                        roots.append(candidate)
            if len(roots) != 1:
                raise ValueError(f"Expected one validated Full9 assembly after restore, found {roots}")
            FULL9_ROOT = roots[0]

    full9_report = validate_full9_store(FULL9_ROOT, EXPECTED_COUNTS)
    source_records = {}
    for split, expected in EXPECTED_COUNTS.items():
        manifest = validate_feature_store(FULL9_ROOT, split=split, require_labels=split != "test")
        records = _read_source_manifest(FULL9_ROOT / split / "source_manifest.json", split=split,
                                        expected_sha256=manifest["source_manifest_sha256"])
        assert len(records) == expected
        if split == "test" and any(record.get("label_raw") is not None for record in records):
            raise ValueError("Full9 test source manifest must be label-free")
        source_records[split] = records

    print(json.dumps({"full9_root": str(FULL9_ROOT), "splits": full9_report}, indent=2))
    (LOG_ROOT / "full9_verified.json").write_text(json.dumps({"full9_root": str(FULL9_ROOT), "splits": full9_report}, indent=2))


## 05 — Prepare label-free Whisper metadata from the signed Full9 source manifest

Do not use the official `test_average.csv` or test scores. The Whisper dataset only needs WAV basename and exact raw caption. Use a stable manifest-derived row order; no inner joins, reordering, or normalization.

In [ ]:
if NEED_FULL7_BUILD:
    for split, records in source_records.items():
        target = METADATA_ROOT / f"{split}_whisper.csv"
        with target.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=["wav_file_name", "text"])
            writer.writeheader()
            for record in records:
                writer.writerow({"wav_file_name": record["wave_file_name"], "text": record["raw_caption"]})
        with target.open(encoding="utf-8", newline="") as handle:
            assert sum(1 for _ in csv.DictReader(handle)) == len(records)
        print(split, "metadata:", target, "rows:", len(records))


## 06 — Train or verify E5 (Set A) using official validation

E5 learns the residual head over Top2 M2D + MS-CLAP + MGA features. No backbones and no Whisper. It trains on all official train rows and evaluates, selects, and early-stops on all official validation rows.

In [ ]:
import yaml
from data.set_a_cached_dataset import SetACachedDataset
from model.expert_adapters.set_a_residual_adapter import SetAResidualAdapter
from model.composite_experts.set_a_residual import SetAResidualHead
from training.set_a_trainer import load_set_a_checkpoint


def run_logged(name, cmd, *, env=None):
    target = LOG_ROOT / f"{name}.log"
    print("RUN", name, "=>", target, flush=True)
    try:
        with target.open("w", encoding="utf-8") as output:
            subprocess.run(
                [str(value) for value in cmd], cwd=REPO_ROOT, env=env,
                stdout=output, stderr=subprocess.STDOUT, check=True,
            )
    except subprocess.CalledProcessError:
        print("FAILED", name, "log tail:", target.read_text(errors="replace")[-5000:])
        raise
    print("PASS", name, "tail:", target.read_text(errors="replace")[-1200:], flush=True)


def gpu_env(physical_gpu):
    environment = os.environ.copy()
    environment["CUDA_VISIBLE_DEVICES"] = str(physical_gpu)
    environment["TOKENIZERS_PARALLELISM"] = "false"
    return environment


def e5_training_config(config):
    training = config["training"]
    return {
        "batch_size": training["batch_size"],
        "num_workers": training["num_workers"],
        "learning_rate": training["learning_rate"],
        "weight_decay": training["weight_decay"],
        "gradient_clip": training["gradient_clip"],
        "selection_metric": "official_validation_mse",
        "selection_policy": "official_validation_mse_v1",
        "early_stopping_patience": training["early_stopping_patience"],
        "min_delta": training["min_delta"],
    }


def e5_checkpoint_matches(path, role, train_dataset, validation_dataset, best_path):
    path = Path(path)
    if not path.is_file():
        return False
    try:
        model = SetAResidualAdapter(SetAResidualHead(**e5_cfg["model"]))
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=e5_cfg["training"]["learning_rate"],
            weight_decay=e5_cfg["training"]["weight_decay"],
        )
        checkpoint = load_set_a_checkpoint(
            path, model=model, optimizer=optimizer,
            expected_dataset=train_dataset,
            expected_data_provenance=train_dataset.provenance,
            map_location="cpu",
        )
        state = checkpoint["training_state"]
        expected_role = {"best": "best", "final_best": "final_best", "latest": "latest"}[role]
        if (checkpoint.get("format_version") != 3
                or checkpoint.get("checkpoint_role") != expected_role
                or state.get("selection_policy") != "official_validation_mse_v1"
                or state.get("training_config") != e5_training_config(e5_cfg)
                or state.get("train_provenance") != train_dataset.provenance
                or state.get("validation_provenance") != validation_dataset.provenance
                or state.get("train_sample_ids") != list(train_dataset.sample_ids)
                or state.get("validation_sample_ids") != list(validation_dataset.sample_ids)
                or checkpoint.get("data_provenance") != train_dataset.provenance):
            return False
        if role == "latest":
            if (not best_path.is_file()
                    or state.get("best_checkpoint_sha256") != sha256_file(best_path)
                    or checkpoint.get("epoch", 0) >= e5_cfg["training"]["epochs"]):
                return False
        return True
    except Exception as exc:
        print("E5 checkpoint incompatible:", path, f"{type(exc).__name__}: {exc}")
        return False


if NEED_FULL7_BUILD:
    e5_cfg = yaml.safe_load((REPO_ROOT / "configs/set_a_residual.example.yaml").read_text(encoding="utf-8"))
    e5_cfg["feature_store_root"] = str(FULL9_ROOT)
    e5_cfg["training"]["selection_metric"] = "official_validation_mse"
    e5_cfg["runtime"]["device"] = "cuda"

    e5_data = SetACachedDataset(FULL9_ROOT, split="train", require_labels=True)
    e5_validation_data = SetACachedDataset(FULL9_ROOT, split="validation", require_labels=True)
    if (len(e5_data), len(e5_validation_data)) != (EXPECTED_COUNTS["train"], EXPECTED_COUNTS["validation"]):
        raise ValueError("E5 requires all 7,500 official train and 3,000 validation rows")

    E5_CONFIG = CONFIG_ROOT / "e5_full7.yaml"
    E5_SELECTION_POLICY = "official_validation_mse_v1"
    E5_BEST = E5_ROOT / "expert5_best.pt"
    E5_FINAL = E5_ROOT / "expert5_final.pt"
    E5_LATEST = E5_ROOT / "expert5_latest.pt"
    e5_compatibility = {}
    for checkpoint_path, checkpoint_role in ((E5_BEST, "best"), (E5_FINAL, "final_best")):
        e5_compatibility[checkpoint_role] = e5_checkpoint_matches(
            checkpoint_path, checkpoint_role, e5_data, e5_validation_data, E5_BEST,
        )
    best_compatible = e5_compatibility["best"]
    final_compatible = e5_compatibility["final_best"]
    latest_compatible = e5_checkpoint_matches(E5_LATEST, "latest", e5_data, e5_validation_data, E5_BEST)
    e5_complete = best_compatible and final_compatible
    e5_resume_pair = best_compatible and latest_compatible and not e5_complete
    e5_has_artifacts = any(path.exists() for path in (E5_BEST, E5_FINAL, E5_LATEST))
    if e5_has_artifacts and not e5_complete and not e5_resume_pair:
        previous_root = E5_ROOT
        E5_ROOT = E5_ROOT_BASE / f"{E5_ROOT_BASE.name}_fresh_{time.time_ns()}"
        print("Incompatible E5 artifacts preserved at", previous_root, "; starting fresh E5 training at", E5_ROOT)

    def write_e5_config():
        e5_cfg["output_dir"] = str(E5_ROOT)
        E5_CONFIG.write_text(yaml.safe_dump(e5_cfg, sort_keys=False), encoding="utf-8")

    write_e5_config()
    E5_BEST = E5_ROOT / "expert5_best.pt"
    E5_FINAL = E5_ROOT / "expert5_final.pt"
    E5_LATEST = E5_ROOT / "expert5_latest.pt"
    if not e5_complete and not RUN_E5_TRAIN:
        raise FileNotFoundError("No compatible official-validation E5 best/final pair; enable RUN_E5_TRAIN.")

    del e5_data, e5_validation_data
    gc.collect()
    if RUN_E5_TRAIN and not e5_complete:
        command = [sys.executable, REPO_ROOT / "scripts/train_set_a_residual.py", "--config", E5_CONFIG]
        if e5_resume_pair:
            command += ["--resume", E5_LATEST]
        run_logged("e5_train", command, env=gpu_env(E5_DEVICE))

    e5_data = SetACachedDataset(FULL9_ROOT, split="train", require_labels=True)
    e5_validation_data = SetACachedDataset(FULL9_ROOT, split="validation", require_labels=True)
    if (not e5_checkpoint_matches(E5_BEST, "best", e5_data, e5_validation_data, E5_BEST)
            or not e5_checkpoint_matches(E5_FINAL, "final_best", e5_data, e5_validation_data, E5_BEST)):
        raise ValueError("E5 best/final checkpoints failed full official train/validation compatibility checks")
    print("E5 official split counts:", len(e5_data), len(e5_validation_data))
    print("E5 verified best/final:", E5_BEST, E5_FINAL)
    del e5_data, e5_validation_data
    gc.collect()


## 07 — Resolve and verify the complete Whisper-large-v2 snapshot

Both GPU workers read the same immutable snapshot. If Internet is off, attach a complete local Whisper-large-v2 snapshot as a Kaggle Input and set `WHISPER_SNAPSHOT_INPUT`. Never switch the Whisper model variant. The `whisper_top2` runtime records checkpoint file hashes.

In [ ]:
if NEED_FULL7_BUILD:
    from huggingface_hub import snapshot_download
    from model.feature_extractors.whisper_top2 import _local_model_files

    WHISPER_REQUIRED_ASSETS = [
        "config.json", "preprocessor_config.json", "tokenizer_config.json",
        "vocab.json", "merges.txt",
    ]
    WHISPER_WEIGHT_PATTERNS = [
        "model.safetensors", "model.safetensors.index.json", "model-*.safetensors",
        "pytorch_model.bin", "pytorch_model.bin.index.json", "pytorch_model-*.bin",
    ]
    WHISPER_ALLOW_PATTERNS = [
        *WHISPER_REQUIRED_ASSETS,
        "tokenizer.json", "special_tokens_map.json", "added_tokens.json",
        *WHISPER_WEIGHT_PATTERNS,
    ]

    def verify_whisper_snapshot(path, *, local_model_files_fn=None):
        path = Path(path).expanduser().resolve(strict=True)
        if local_model_files_fn is None:
            local_model_files_fn = _local_model_files
        files = local_model_files_fn(path)
        required_names = {item.name for item in files}
        missing = [name for name in WHISPER_REQUIRED_ASSETS if name not in required_names]
        if missing:
            raise FileNotFoundError(f"Whisper-large-v2 snapshot is missing required assets: {missing}")
        weight_files = [
            item for item in files
            if item.name.endswith((".safetensors", ".bin"))
        ]
        if not weight_files or any(not item.is_file() or item.stat().st_size <= 0 for item in weight_files):
            raise FileNotFoundError("Whisper-large-v2 snapshot has no complete, non-empty checkpoint weights")
        return files

    def resolve_whisper_snapshot(
        snapshot_input, *, allow_download, cache_dir,
        snapshot_download_fn=None, local_model_files_fn=None,
    ):
        if type(allow_download) is not bool:
            raise TypeError("allow_download must be boolean")
        if snapshot_download_fn is None:
            snapshot_download_fn = snapshot_download
        cache_dir = Path(cache_dir).expanduser().resolve()
        cache_dir.mkdir(parents=True, exist_ok=True)
        if snapshot_input is not None:
            selected = Path(snapshot_input).expanduser().resolve(strict=True)
            verify_whisper_snapshot(selected, local_model_files_fn=local_model_files_fn)
            return selected

        options = {
            "repo_id": "openai/whisper-large-v2",
            "cache_dir": str(cache_dir),
            "allow_patterns": WHISPER_ALLOW_PATTERNS,
        }
        try:
            selected = Path(snapshot_download_fn(**options, local_files_only=True))
            verify_whisper_snapshot(selected, local_model_files_fn=local_model_files_fn)
            return selected.resolve(strict=True)
        except Exception as cache_error:
            if not allow_download:
                raise FileNotFoundError(
                    "A complete local Whisper-large-v2 snapshot with checkpoint weights is required; "
                    "attach it or enable ALLOW_WHISPER_DOWNLOAD."
                ) from cache_error
        selected = Path(snapshot_download_fn(**options, local_files_only=False))
        verify_whisper_snapshot(selected, local_model_files_fn=local_model_files_fn)
        return selected.resolve(strict=True)

    snapshot_path = resolve_whisper_snapshot(
        WHISPER_SNAPSHOT_INPUT, allow_download=ALLOW_WHISPER_DOWNLOAD,
        cache_dir=WHISPER_CACHE_DIR,
    )
    whisper_model_ref = str(snapshot_path)
    WHISPER_CONFIG = CONFIG_ROOT / "whisper_full7.yaml"
    whisper_cfg = yaml.safe_load((REPO_ROOT / "configs/whisper_top2.example.yaml").read_text(encoding="utf-8"))
    whisper_cfg["whisper_top2"]["model_name_or_path"] = whisper_model_ref
    whisper_cfg["runtime"].update({
        "device": "auto", "offline": True, "cache_dir": str(WHISPER_CACHE_DIR),
    })
    WHISPER_CONFIG.write_text(yaml.safe_dump(whisper_cfg, sort_keys=False), encoding="utf-8")
    print("Verified immutable Whisper-large-v2 snapshot:", snapshot_path)
    print("Verified config, tokenizer, and PyTorch checkpoint assets:", len(_local_model_files(snapshot_path)))


## 08 — Whisper smoke tests on both physical T4 GPUs

Each worker has `CUDA_VISIBLE_DEVICES` set to one physical GPU, and internally uses logical `cuda:0`; this avoids real E6 cache metadata differing between train and validation. Always use batch size **1**, including resumed runs.

In [ ]:
if NEED_FULL7_BUILD:
    def whisper_command(split, *, smoke=False):
        cmd = [sys.executable, str(REPO_ROOT / "scripts/extract_whisper_features.py"),
               "--config", str(WHISPER_CONFIG),
               "--metadata-csv", str(METADATA_ROOT / f"{split}_whisper.csv"),
               "--audio-dir", str(DATA_ROOT / "wav" / split),
               "--split", split,
               "--output-dir", str(WHISPER_ROOT),
               "--batch-size", str(WHISPER_BATCH_SIZE),
               "--num-workers", str(WHISPER_NUM_WORKERS)]
        if smoke:
            cmd += ["--smoke-test"]
        return cmd

    if RUN_WHISPER_SMOKE:
        run_logged("whisper_train_gpu0_smoke", whisper_command("train", smoke=True), env=gpu_env(0))
        run_logged("whisper_validation_gpu1_smoke", whisper_command("validation", smoke=True), env=gpu_env(1))
    print("Whisper smoke complete. Verify [1,1280] audio and text shapes in both logs.")


## 09 — Dual-GPU Whisper extraction with verified resumable caches

GPU 0 extracts **train**. GPU 1 extracts **validation then test**, sequentially, while train runs independently. Two *subprocesses* keep GPU contexts isolated. Each split writes only to `whisper_cache/<split>` and supports resuming a validated prefix. Log files can be inspected if Kaggle stops a worker.

In [ ]:
if NEED_FULL7_BUILD:
    def whisper_command(split, *, smoke=False):
        command = [
            sys.executable, str(REPO_ROOT / "scripts/extract_whisper_features.py"),
            "--config", str(WHISPER_CONFIG),
            "--metadata-csv", str(METADATA_ROOT / f"{split}_whisper.csv"),
            "--audio-dir", str(DATA_ROOT / "wav" / split),
            "--split", split,
            "--output-dir", str(WHISPER_ROOT),
            "--batch-size", str(WHISPER_BATCH_SIZE),
            "--num-workers", str(WHISPER_NUM_WORKERS),
        ]
        if smoke:
            command += ["--smoke-test"]
        return command

    def start_whisper_worker(split, physical_gpu):
        log_path = LOG_ROOT / f"whisper_{split}_gpu{physical_gpu}.log"
        stream = log_path.open("a", encoding="utf-8")
        stream.write(f"\n== START {split} on physical GPU {physical_gpu}: {time.ctime()} ==\n")
        stream.flush()
        try:
            process = subprocess.Popen(
                [str(value) for value in whisper_command(split)],
                cwd=REPO_ROOT, env=gpu_env(physical_gpu),
                stdout=stream, stderr=subprocess.STDOUT,
            )
        except BaseException:
            stream.close()
            raise
        return process, stream, log_path

    def run_whisper_workers(start_worker, *, poll_interval=1.0, heartbeat_seconds=60.0,
                            terminate_timeout=10.0):
        if min(poll_interval, heartbeat_seconds, terminate_timeout) <= 0:
            raise ValueError("Whisper worker intervals must be positive")
        active = {}
        all_workers = {}
        statuses = {}
        last_heartbeat = time.monotonic()

        def start(name, gpu):
            process, stream, log_path = start_worker(name, gpu)
            record = {
                "process": process, "stream": stream, "log_path": Path(log_path),
                "started": time.monotonic(),
            }
            active[name] = record
            all_workers[name] = record

        def close_worker(name, return_code):
            record = active.pop(name)
            statuses[name] = return_code
            try:
                record["stream"].close()
            except Exception:
                pass
            if return_code != 0:
                raise RuntimeError(
                    f"Whisper {name} extraction exited with status {return_code}; "
                    f"log: {record['log_path']}"
                )

        def poll_workers(required_active):
            nonlocal last_heartbeat
            while required_active in active:
                for name in list(active):
                    return_code = active[name]["process"].poll()
                    if return_code is not None:
                        close_worker(name, return_code)
                if required_active not in active:
                    return
                now = time.monotonic()
                if now - last_heartbeat >= heartbeat_seconds:
                    progress = {
                        name: int(now - record["started"])
                        for name, record in active.items()
                    }
                    print("WHISPER HEARTBEAT active_seconds:", progress, flush=True)
                    last_heartbeat = now
                time.sleep(poll_interval)

        def terminate_active():
            for record in active.values():
                process = record["process"]
                try:
                    if process.poll() is None:
                        process.terminate()
                except Exception:
                    pass
            for name, record in list(active.items()):
                process = record["process"]
                try:
                    process.wait(timeout=terminate_timeout)
                except subprocess.TimeoutExpired:
                    process.kill()
                    try:
                        process.wait(timeout=terminate_timeout)
                    except subprocess.TimeoutExpired:
                        print("Could not reap terminated Whisper worker:", name, flush=True)
                except Exception:
                    try:
                        process.kill()
                    except Exception:
                        pass
                try:
                    record["stream"].close()
                except Exception:
                    pass
            active.clear()

        try:
            start("train", 0)
            start("validation", 1)
            # Observe train while validation runs; launch test on GPU 1 as soon as
            # validation succeeds, even if the independent train worker continues.
            poll_workers("validation")
            start("test", 1)
            while active:
                for name in list(active):
                    return_code = active[name]["process"].poll()
                    if return_code is not None:
                        close_worker(name, return_code)
                if active:
                    now = time.monotonic()
                    if now - last_heartbeat >= heartbeat_seconds:
                        progress = {
                            name: int(now - record["started"])
                            for name, record in active.items()
                        }
                        print("WHISPER HEARTBEAT active_seconds:", progress, flush=True)
                        last_heartbeat = now
                    time.sleep(poll_interval)
            if statuses != {"train": 0, "validation": 0, "test": 0}:
                raise RuntimeError(f"Incomplete Whisper worker status set: {statuses}")
            return statuses
        except BaseException:
            terminate_active()
            for name, record in all_workers.items():
                log_path = record["log_path"]
                if log_path.is_file():
                    print(
                        f"Whisper {name} log tail:\n"
                        f"{log_path.read_text(errors='replace')[-5000:]}",
                        flush=True,
                    )
            raise

    if RUN_WHISPER_SMOKE:
        run_logged("whisper_train_gpu0_smoke", whisper_command("train", smoke=True), env=gpu_env(0))
        run_logged("whisper_validation_gpu1_smoke", whisper_command("validation", smoke=True), env=gpu_env(1))
    print("Whisper smoke complete; original batch size remains 1.")

    if RUN_WHISPER_EXTRACT:
        whisper_statuses = run_whisper_workers(start_whisper_worker)
        print("Whisper split worker exit codes:", whisper_statuses)
    else:
        print("Whisper extraction disabled; all existing split caches must pass full verification before E6.")


## 10 — Verify all Whisper splits and Full9-to-Whisper alignment

This step reads the actual original WAV bytes and tests their hashes through `SetBCachedDataset`, verifies order and 1280+1280 dimensions, and catches GPU/preprocessing provenance divergence before E6 training. Do not patch the cache if validation fails.

In [ ]:
if NEED_FULL7_BUILD:
    from data.set_b_cached_dataset import SetBCachedDataset
    from data.whisper_feature_cache import FEATURE_DIMENSIONS, FEATURE_EXTRACTION_VERSION
    from data.audio_overlap import build_train_validation_audio_overlap_diagnostic

    whisper_cache_summaries = {}
    e6_expected_provenance = {}
    e6_expected_membership = {}
    e6_audio_records = {}
    shared_checkpoint_identity = None
    shared_preprocessing_identity = None

    for split, expected in EXPECTED_COUNTS.items():
        dataset = SetBCachedDataset(
            FULL9_ROOT, WHISPER_ROOT, split=split,
            source_audio_dir=DATA_ROOT / "wav" / split,
        )
        if len(dataset) != expected:
            raise ValueError(f"Whisper {split}: expected {expected} aligned Full9 rows, found {len(dataset)}")
        expected_ids = list(dataset.sample_ids)
        cached_ids = list(dataset.whisper_reader.sample_ids)
        if (cached_ids != expected_ids or len(set(cached_ids)) != expected
                or dataset.store.sample_ids != expected_ids):
            raise ValueError(f"Whisper {split}: cache sample IDs/order differ from the exact Full9 source rows")
        if (dataset.whisper_audio.shape != (expected, 1280)
                or dataset.whisper_text.shape != (expected, 1280)
                or dataset.whisper_audio.dtype != np.float32
                or dataset.whisper_text.dtype != np.float32
                or not np.isfinite(dataset.whisper_audio).all()
                or not np.isfinite(dataset.whisper_text).all()):
            raise ValueError(f"Whisper {split}: expected finite float32 audio/text arrays [N,1280]")

        cache_manifest_path = WHISPER_ROOT / split / "manifest.json"
        cache_manifest = json.loads(cache_manifest_path.read_text(encoding="utf-8"))
        shard_ids = [
            sample_id for shard in cache_manifest["shards"]
            for sample_id in shard["sample_ids"]
        ]
        if (cache_manifest.get("split") != split
                or cache_manifest.get("feature_extraction_version") != FEATURE_EXTRACTION_VERSION
                or cache_manifest.get("feature_dimensions") != FEATURE_DIMENSIONS
                or cache_manifest.get("dataset_identity") != dataset.whisper_reader.dataset_identity
                or cache_manifest.get("source_checkpoint_identity") != dataset.whisper_checkpoint_identity
                or cache_manifest.get("preprocessing_identity") != dataset.whisper_preprocessing_identity
                or sum(shard["sample_count"] for shard in cache_manifest["shards"]) != expected
                or shard_ids != expected_ids):
            raise ValueError(f"Whisper {split}: manifest, shard counts, identities or row membership mismatch")

        if shared_checkpoint_identity is None:
            shared_checkpoint_identity = dataset.whisper_checkpoint_identity
            shared_preprocessing_identity = dataset.whisper_preprocessing_identity
        elif (dataset.whisper_checkpoint_identity != shared_checkpoint_identity
                or dataset.whisper_preprocessing_identity != shared_preprocessing_identity):
            raise ValueError("Whisper checkpoint/preprocessing identities differ across splits")


    # E6 checkpoint metadata contains TRAIN and VALIDATION only.
    # The TEST Whisper cache is verified separately for Full7/Full8
    # inference, but must not become part of E6 training provenance.

    if split in {"train", "validation"}:

        e6_expected_provenance[split] = dataset.provenance

        e6_audio_records[split] = list(dataset.records)

        e6_expected_membership[split] = {
            "sample_count": expected,
            "ordered_sample_ids_sha256": hashlib.sha256(
                json.dumps(
                    expected_ids,
                    ensure_ascii=False,
                    separators=(",", ":"),
                ).encode("utf-8")
            ).hexdigest(),
            "source_manifest_sha256": dataset.provenance[
                "full9_source_manifest_sha256"
            ],
        }

        whisper_cache_summaries[split] = {
            "sample_count": expected,
            "sample_ids": expected_ids,
            "dataset_identity": dataset.whisper_reader.dataset_identity,
            "source_checkpoint_identity": dataset.whisper_checkpoint_identity,
            "preprocessing_identity": dataset.whisper_preprocessing_identity,
            "manifest_sha256": sha256_file(cache_manifest_path),
        }
        print(
            f"Whisper {split} PASS: {expected} samples, "
            f"{len(cache_manifest['shards'])} verified shards, "
            f"checkpoint={cache_manifest_path}",
        )
        del dataset
        gc.collect()


    expected_e6_splits = {"train", "validation"}
    expected_whisper_splits = {"train", "validation", "test"}

    if set(e6_expected_provenance) != expected_e6_splits:
        raise ValueError(
            "E6 provenance must contain exactly train and validation. "
            f"Found: {sorted(e6_expected_provenance)}"
        )

    if set(e6_expected_membership) != expected_e6_splits:
        raise ValueError(
            "E6 membership must contain exactly train and validation. "
            f"Found: {sorted(e6_expected_membership)}"
        )

    if set(e6_audio_records) != expected_e6_splits:
        raise ValueError(
            "E6 audio records must contain exactly train and validation."
        )

    if set(whisper_cache_summaries) != expected_whisper_splits:
        raise ValueError(
            "Whisper cache verification must cover all three splits. "
            f"Found: {sorted(whisper_cache_summaries)}"
        )

    for split in expected_e6_splits:
        if (
            e6_expected_membership[split]["sample_count"]
            != EXPECTED_COUNTS[split]
        ):
            raise ValueError(
                f"E6 {split} membership count mismatch"
            )

    print("E6 training provenance: PASS")
    print("E6 membership: PASS")
    print("Whisper train/validation/test cache coverage: PASS")

    e6_expected_overlap_diagnostic = build_train_validation_audio_overlap_diagnostic(
        e6_audio_records["train"], e6_audio_records["validation"],
        allow_train_validation_audio_overlap=ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP,
    )
    if (e6_expected_overlap_diagnostic["overlapping_sha256_count"]
            and not ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP):
        raise ValueError("E6 train/validation WAV overlap is forbidden by the configured policy")

    def archive_verified_whisper_cache(cache_root, archive_path, manifest_path, split_summaries):
        cache_root = Path(cache_root).expanduser().resolve(strict=True)
        archive_path = Path(archive_path).expanduser().resolve()
        manifest_path = Path(manifest_path).expanduser().resolve()
        if set(split_summaries) != {"train", "validation", "test"}:
            raise ValueError("Whisper archive requires exactly train, validation and test cache summaries")

        def digest_file(path):
            digest = hashlib.sha256()
            with Path(path).open("rb") as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b""):
                    digest.update(block)
            return digest.hexdigest()

        expected_files = {}
        validated_splits = {}
        for split in ("train", "validation", "test"):
            summary = split_summaries[split]
            split_root = cache_root / split
            manifest_path_split = split_root / "manifest.json"
            if not manifest_path_split.is_file():
                raise FileNotFoundError(f"Whisper {split} cache manifest is missing")
            raw_manifest = manifest_path_split.read_bytes()
            if digest_file(manifest_path_split) != summary["manifest_sha256"]:
                raise ValueError(f"Whisper {split} manifest changed after cache validation")
            cache_manifest = json.loads(raw_manifest)
            body = {key: value for key, value in cache_manifest.items() if key != "metadata_sha256"}
            expected_metadata_sha = hashlib.sha256(json.dumps(
                body, sort_keys=True, separators=(",", ":"), ensure_ascii=False, allow_nan=False,
            ).encode("utf-8")).hexdigest()
            if (cache_manifest.get("metadata_sha256") != expected_metadata_sha
                    or cache_manifest.get("split") != split
                    or cache_manifest.get("dataset_identity") != summary["dataset_identity"]
                    or cache_manifest.get("source_checkpoint_identity") != summary["source_checkpoint_identity"]
                    or cache_manifest.get("preprocessing_identity") != summary["preprocessing_identity"]
                    or cache_manifest.get("feature_dimensions") != {"audio": 1280, "text": 1280, "total": 2560}
                    or cache_manifest.get("dataset_identity", {}).get("sample_count") != summary["sample_count"]):
                raise ValueError(f"Whisper {split} manifest identity changed before archive")
            shard_ids = []
            split_files = [manifest_path_split]
            for shard in cache_manifest.get("shards", []):
                filename = shard.get("file")
                if not isinstance(filename, str) or Path(filename).name != filename or not filename.endswith(".npz"):
                    raise ValueError(f"Whisper {split} shard filename is unsafe")
                shard_path = split_root / filename
                if not shard_path.is_file() or digest_file(shard_path) != shard.get("sha256"):
                    raise ValueError(f"Whisper {split} shard SHA-256 mismatch: {shard_path}")
                with np.load(shard_path, allow_pickle=False) as arrays:
                    if set(arrays.files) != {"sample_ids", "whisper_audio_features", "whisper_text_features"}:
                        raise ValueError(f"Whisper shard arrays are invalid: {shard_path}")
                    ids = [str(value) for value in arrays["sample_ids"].tolist()]
                    audio, text = arrays["whisper_audio_features"], arrays["whisper_text_features"]
                    if (ids != shard.get("sample_ids") or len(ids) != shard.get("sample_count")
                            or audio.dtype != np.float32 or text.dtype != np.float32
                            or audio.shape != (len(ids), 1280) or text.shape != (len(ids), 1280)
                            or not np.isfinite(audio).all() or not np.isfinite(text).all()):
                        raise ValueError(f"Whisper shard shape, dtype, finite values or sample IDs are invalid: {shard_path}")
                shard_ids.extend(ids)
                split_files.append(shard_path)
            if shard_ids != summary["sample_ids"] or len(shard_ids) != summary["sample_count"]:
                raise ValueError(f"Whisper {split} archive membership does not match verified source order")
            for file_path in split_files:
                relative = file_path.relative_to(cache_root).as_posix()
                expected_files[relative] = {
                    "sha256": digest_file(file_path),
                    "size_bytes": file_path.stat().st_size,
                }
            validated_splits[split] = {
                "sample_count": summary["sample_count"],
                "dataset_identity": summary["dataset_identity"],
                "source_checkpoint_identity": summary["source_checkpoint_identity"],
                "preprocessing_identity": summary["preprocessing_identity"],
                "manifest_sha256": summary["manifest_sha256"],
                "shard_count": len(cache_manifest["shards"]),
            }

        actual_files = {}
        for file_path in cache_root.rglob("*"):
            if file_path.is_symlink():
                raise ValueError(f"Whisper cache contains a symlink: {file_path}")
            if file_path.is_file():
                actual_files[file_path.relative_to(cache_root).as_posix()] = file_path
        if set(actual_files) != set(expected_files):
            raise ValueError("Whisper cache contains missing or unreferenced files; refusing archive")

        archive_path.parent.mkdir(parents=True, exist_ok=True)
        descriptor, temporary_name = tempfile.mkstemp(
            prefix=archive_path.name + ".", suffix=".tmp", dir=archive_path.parent,
        )
        try:
            with os.fdopen(descriptor, "wb") as output:
                with tarfile.open(fileobj=output, mode="w") as archive:
                    for relative in sorted(expected_files):
                        archive.add(actual_files[relative], arcname=f"whisper_cache/{relative}", recursive=False)
                output.flush()
                os.fsync(output.fileno())

            expected_members = {
                f"whisper_cache/{relative}": info for relative, info in expected_files.items()
            }
            observed_members = {}
            with tarfile.open(temporary_name, "r:") as archive:
                for member in archive.getmembers():
                    if not member.isfile() or member.name in observed_members:
                        raise ValueError(f"Whisper archive has an unsafe or duplicate member: {member.name}")
                    expected = expected_members.get(member.name)
                    if expected is None:
                        raise ValueError(f"Whisper archive contains an unverified member: {member.name}")
                    reader = archive.extractfile(member)
                    digest = hashlib.sha256()
                    size = 0
                    while True:
                        chunk = reader.read(1024 * 1024)
                        if not chunk:
                            break
                        digest.update(chunk)
                        size += len(chunk)
                    if digest.hexdigest() != expected["sha256"] or size != expected["size_bytes"]:
                        raise ValueError(f"Whisper archive member failed SHA-256 verification: {member.name}")
                    observed_members[member.name] = expected
            if observed_members != expected_members:
                raise ValueError("Whisper archive is incomplete")
            os.replace(temporary_name, archive_path)
        finally:
            Path(temporary_name).unlink(missing_ok=True)

        report = {
            "schema_version": 1,
            "status": "PASS",
            "archive_name": archive_path.name,
            "archive_sha256": digest_file(archive_path),
            "archive_size_bytes": archive_path.stat().st_size,
            "files": [
                {"path": name, **expected_files[name]}
                for name in sorted(expected_files)
            ],
            "splits": validated_splits,
            "persistence_scope": (
                "Local /kaggle/working output for this Kaggle session. "
                "Save a Kaggle notebook output version or download it to retain across sessions."
            ),
            "upload_performed": False,
        }
        report_body = dict(report)
        report["manifest_sha256"] = hashlib.sha256(json.dumps(
            report_body, sort_keys=True, separators=(",", ":"), ensure_ascii=False, allow_nan=False,
        ).encode("utf-8")).hexdigest()
        atomic_write_json(manifest_path, report)
        return report

    WHISPER_ARCHIVE_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    whisper_archive_report = archive_verified_whisper_cache(
        WHISPER_ROOT, WHISPER_ARCHIVE, WHISPER_ARCHIVE_MANIFEST, whisper_cache_summaries,
    )
    print(
        "Verified local Whisper cache archive:",
        WHISPER_ARCHIVE,
        whisper_archive_report["archive_sha256"],
    )
    print(
        "This archive is local /kaggle/working session output only. Save a Kaggle output version "
        "or download it for cross-session persistence. No dataset upload is performed."
    )


## 11 — Train or verify E6 (Set B), then persist the verified Whisper cache

E6 has a fixed 30-epoch training policy; its existing code selects the best checkpoint by validation MSE. This behavior is **provisional for challenge compliance** and must be clarified with organizers. Use the existing checkpoint loader, not a hand-built substitute.

In [ ]:
if NEED_FULL7_BUILD:
    E6_CONFIG = CONFIG_ROOT / "e6_full7.yaml"
    e6_cfg = yaml.safe_load((REPO_ROOT / "configs/set_b_residual.example.yaml").read_text(encoding="utf-8"))
    e6_cfg["feature_store_root"] = str(FULL9_ROOT)
    e6_cfg["whisper_cache_root"] = str(WHISPER_ROOT)
    e6_cfg["runtime"]["device"] = "cuda"
    e6_cfg["training"]["allow_train_validation_audio_overlap"] = ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP
    e6_cfg["source_audio_dirs"] = {split: str(DATA_ROOT / "wav" / split) for split in EXPECTED_COUNTS}
    e6_cfg["source_manifest_paths"] = {}

    E6_ROOT = E6_ROOT_BASE
    E6_BEST = E6_ROOT / "expert6_best.pt"
    E6_LATEST = E6_ROOT / "expert6_latest.pt"
    E6_FINAL = E6_ROOT / "expert6_final.pt"
    from data.set_b_cached_dataset import SetBCachedDataset
    from data.audio_overlap import build_train_validation_audio_overlap_diagnostic
    from model.composite_experts.set_b_residual import SetBResidualHead
    from model.expert_adapters.set_b_residual_adapter import SetBResidualAdapter
    from training.set_b_trainer import load_set_b_checkpoint

    e6_expected_overlap_diagnostic = build_train_validation_audio_overlap_diagnostic(
        e6_audio_records["train"], e6_audio_records["validation"],
        allow_train_validation_audio_overlap=ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP,
    )
    e6_training = e6_cfg["training"]
    e6_expected_settings = {
        "epochs": 30,
        "batch_size": e6_training["batch_size"],
        "num_workers": e6_training["num_workers"],
        "learning_rate": e6_training["learning_rate"],
        "weight_decay": e6_training["weight_decay"],
        "gradient_clip": e6_training["gradient_clip"],
        "seed": e6_training["seed"],
        "selection_metric": "official_validation_mse",
        "selection_policy": "official_validation_mse_min_v1",
        "allow_train_validation_audio_overlap": ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP,
    }
    e6_train_dataset = SetBCachedDataset(
        FULL9_ROOT, WHISPER_ROOT, split="train",
        source_manifest_path=e6_cfg["source_manifest_paths"].get("train"),
        source_audio_dir=e6_cfg["source_audio_dirs"]["train"],
    )
    if len(e6_train_dataset) != EXPECTED_COUNTS["train"]:
        raise ValueError("E6 compatibility requires all 7,500 official training rows")

    def e6_checkpoint_matches_experiment(path, role):
        path = Path(path)
        if not path.is_file():
            return False
        try:
            model = SetBResidualAdapter(SetBResidualHead(**e6_cfg["model"]))
            optimizer = torch.optim.AdamW(
                model.parameters(), lr=e6_training["learning_rate"],
                weight_decay=e6_training["weight_decay"],
            )
            checkpoint = load_set_b_checkpoint(
                path, model=model, optimizer=optimizer,
                expected_dataset=e6_train_dataset, map_location="cpu",
                for_resume=(role == "latest"),
            )
            if (checkpoint.get("format_version") != 2
                    or checkpoint.get("checkpoint_role") != role
                    or checkpoint.get("training_settings") != e6_expected_settings
                    or checkpoint.get("dataset_provenance") != e6_expected_provenance
                    or checkpoint.get("training_membership") != e6_expected_membership
                    or checkpoint.get("audio_overlap_diagnostic") != e6_expected_overlap_diagnostic
                    or checkpoint.get("audio_overlap_diagnostic_sha256")
                        != e6_expected_overlap_diagnostic["diagnostic_sha256"]
                    or checkpoint.get("selection_policy") != "official_validation_mse_min_v1"):
                return False
            if role == "final":
                return (checkpoint.get("epoch") == 30
                        and checkpoint.get("best_checkpoint_sha256") == sha256_file(E6_BEST))
            if role == "latest":
                return (1 <= checkpoint.get("epoch", 0) <= 30
                        and checkpoint.get("best_checkpoint_sha256") == sha256_file(E6_BEST))
            return role == "best"
        except Exception as exc:
            print("E6 checkpoint incompatible:", path, f"{type(exc).__name__}: {exc}")
            return False

    e6_best_compatible = e6_checkpoint_matches_experiment(E6_BEST, "best")
    e6_final_compatible = e6_checkpoint_matches_experiment(E6_FINAL, "final")
    e6_latest_compatible = e6_checkpoint_matches_experiment(E6_LATEST, "latest")
    try:
        latest_epoch = torch.load(E6_LATEST, map_location="cpu", weights_only=False).get("epoch")
    except Exception:
        latest_epoch = None
    e6_complete_for_experiment = (
        e6_best_compatible and e6_final_compatible and e6_latest_compatible
        and latest_epoch == 30
    )
    e6_resume_pair = (
        e6_best_compatible and e6_latest_compatible
        and isinstance(latest_epoch, int) and 0 < latest_epoch < 30
    )
    e6_has_artifacts = any(path.exists() for path in (E6_BEST, E6_LATEST, E6_FINAL))
    if e6_has_artifacts and not e6_complete_for_experiment and not e6_resume_pair:
        previous_root = E6_ROOT
        E6_ROOT = E6_ROOT_BASE.parent / f"{E6_ROOT_BASE.name}_fresh_{time.time_ns()}"
        print("Incompatible E6 artifacts preserved at", previous_root,
              "; starting fresh E6 training at", E6_ROOT)
        E6_BEST = E6_ROOT / "expert6_best.pt"
        E6_LATEST = E6_ROOT / "expert6_latest.pt"
        E6_FINAL = E6_ROOT / "expert6_final.pt"
        e6_resume_pair = False

    e6_cfg["output_dir"] = str(E6_ROOT)
    E6_CONFIG.write_text(yaml.safe_dump(e6_cfg, sort_keys=False), encoding="utf-8")
    del e6_train_dataset
    gc.collect()
    if RUN_E6_TRAIN and not e6_complete_for_experiment:
        command = [sys.executable, REPO_ROOT / "scripts/train_set_b_residual.py", "--config", E6_CONFIG]
        if e6_resume_pair:
            command += ["--resume", E6_LATEST]
        run_logged("e6_train", command, env=gpu_env(E6_DEVICE))
    elif not e6_complete_for_experiment:
        raise FileNotFoundError("No compatible complete E6 run; enable RUN_E6_TRAIN to train or safely resume it.")

    if not all(path.is_file() for path in (E6_BEST, E6_LATEST, E6_FINAL)):
        raise FileNotFoundError("E6 requires compatible expert6_best.pt, expert6_latest.pt and expert6_final.pt")
    e6_train_dataset = SetBCachedDataset(
        FULL9_ROOT, WHISPER_ROOT, split="train",
        source_manifest_path=e6_cfg["source_manifest_paths"].get("train"),
        source_audio_dir=e6_cfg["source_audio_dirs"]["train"],
    )
    if not all((
        e6_checkpoint_matches_experiment(E6_BEST, "best"),
        e6_checkpoint_matches_experiment(E6_LATEST, "latest"),
        torch.load(E6_LATEST, map_location="cpu", weights_only=False).get("epoch") == 30,
        e6_checkpoint_matches_experiment(E6_FINAL, "final"),
    )):
        raise ValueError("E6 checkpoint set failed official split, 30-epoch, overlap-policy or optimizer verification")
    print("E6 verified best/latest/final:", E6_BEST, E6_LATEST, E6_FINAL)
    del e6_train_dataset
    gc.collect()


## 12 — Assemble and validate Full7 (CPU; no backbones)

Uses *exact* existing CLI: Full9 → E5/E6 frozen heads → seven score columns and **29,959 features** in E1…E7 order. It binds source WAV/Whisper/E5/E6 provenance and rejects inconsistent labels or cache fingerprints.

In [ ]:
if NEED_FULL7_BUILD:
    from data.moe_feature_store import validate_feature_store
    FULL7_BUILD_REPORT = LOG_ROOT / "full7_build.json"
    if RUN_FULL7_BUILD:
        run_logged("full7_build", [
            sys.executable, REPO_ROOT / "scripts/build_full7_feature_store.py",
            "--full9-root", FULL9_ROOT,
            "--whisper-root", WHISPER_ROOT,
            "--e5-checkpoint", E5_BEST,
            "--e6-checkpoint", E6_BEST,
            "--source-audio-root", DATA_ROOT / "wav",
            "--output-root", FULL7_ROOT,
            "--split", "train", "--split", "validation", "--split", "test",
            "--shard-size", str(SHARD_SIZE),
            "--report-json", FULL7_BUILD_REPORT,
        ] + (["--allow-train-validation-audio-overlap"]
             if ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP else []))
    for split, count in EXPECTED_COUNTS.items():
        manifest = validate_feature_store(FULL7_ROOT, split=split, require_labels=split != "test")
        actual = sum(shard["count"] for shard in manifest["shards"])
        assert (manifest["expert_profile"], manifest["feature_width"], manifest["score_width"], actual) == ("full_7", 29959, 7, count)
        assert manifest["labels_present"] is (split != "test")
        print(split, "FULL7 PASS", actual, manifest["store_identity_sha256"])


## 13 — Verify all Full7 splits for Full8 assembly


In [ ]:
full7_manifests={}
for split,expected in EXPECTED_COUNTS.items():
    manifest=validate_feature_store(FULL7_ROOT,split=split,require_labels=split!="test")
    count=sum(shard["count"] for shard in manifest["shards"])
    if (manifest.get("expert_profile"),manifest.get("feature_width"),manifest.get("score_width"),count)!=("full_7",29959,7,expected): raise ValueError(f"{split}: Full7 input contract mismatch")
    if manifest["labels_present"] is (split=="test"): raise ValueError(f"{split}: Full7 test store must remain label-free")
    full7_manifests[split]=manifest
    print(split,"Full7 verified",count,manifest["store_identity_sha256"])


## 14 — Frozen E8 extraction with verified resumable shards

Batch size one avoids concurrent GPU-heavy workloads. The trained LoRA and CORAL head are loaded for inference only. Test labels are projected out before dataset construction.


In [ ]:
import yaml
E8_CONFIG = CONFIG_ROOT / "expert8_extraction.yaml"
e8_cfg = yaml.safe_load((REPO_ROOT / "configs/expert8_extraction.example.yaml").read_text(encoding="utf-8"))
e8_cfg["runtime"].update({"device": "cuda:0", "offline": True, "cache_dir": str(HF_E8_CACHE_ROOT)})
e8_cfg["experts"]["expert8"]["base_checkpoint"] = str(E8_BASE_CHECKPOINT)
e8_cfg["experts"]["expert8"]["finetuned_checkpoint"] = str(E8_LORA_CHECKPOINT)
E8_CONFIG.write_text(yaml.safe_dump(e8_cfg, sort_keys=False), encoding="utf-8")
E8_EXTRACTION_REPORT = LOG_ROOT / "expert8_extraction.json"

from scripts.extract_expert8_features import (
    _config as _e8_config, _source_rows as _e8_source_rows,
    _shard_sample_identity as _e8_shard_sample_identity,
    validate_expert8_cache_manifest,
)
from scripts.extract_moe_feature_store import compute_expert_cache_identity
from data.moe_expert_cache import load_expert_shard
from model.expert_adapters.expert8_m2d_clap_runtime import CORAL_FEATURE_DIM


def verify_existing_e8_cache(config_path, dataset_root, output_root, report_path):
    config = _e8_config(config_path, device="cuda:0")
    cache_identity = compute_expert_cache_identity(
        "expert8", config, feature_dim=CORAL_FEATURE_DIM, device=torch.device("cuda:0"),
    )
    checkpoint_provenance = cache_identity["checkpoint_provenance"]
    if (checkpoint_provenance["base_checkpoint"]["sha256"] != E8_BASE_SHA256
            or checkpoint_provenance["finetuned_checkpoint"]["sha256"] != E8_LORA_SHA256):
        raise ValueError("E8 cache identity no longer matches the preflight checkpoint SHA-256 values")
    cache_base = Path(output_root).expanduser().resolve() / "expert8" / cache_identity["cache_id"]
    work_root = Path(output_root).expanduser().resolve() / "source_projection"
    split_reports = {}
    for split, expected_count in EXPECTED_COUNTS.items():
        dataset, records = _e8_source_rows(dataset_root, split, work_root, limit=None)
        try:
            if len(records) != expected_count:
                raise ValueError(f"E8 {split}: expected {expected_count} source rows, found {len(records)}")
            split_root = cache_base / split
            manifest_path = split_root / "manifest.json"
            if not manifest_path.is_file():
                raise FileNotFoundError(f"E8 {split} cache manifest missing: {manifest_path}")
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            validate_expert8_cache_manifest(
                manifest, split=split, expected_records=records, cache_identity=cache_identity,
            )
            expected_start = 0
            observed_count = 0
            for index, shard in enumerate(manifest["shards"]):
                if shard.get("index") != index or shard.get("start") != expected_start:
                    raise ValueError(f"E8 {split}: shard indices/order are not contiguous")
                count = shard.get("count")
                if type(count) is not int or count <= 0:
                    raise ValueError(f"E8 {split}: invalid shard sample count")
                selected = records[expected_start:expected_start + count]
                if len(selected) != count:
                    raise ValueError(f"E8 {split}: shard count exceeds official source membership")
                sample_identity = _e8_shard_sample_identity(split, selected, cache_identity)
                relative_path = shard.get("relative_path")
                if not isinstance(relative_path, str) or Path(relative_path).is_absolute():
                    raise ValueError(f"E8 {split}: unsafe shard path")
                shard_path = (split_root / relative_path).resolve(strict=True)
                if not shard_path.is_relative_to(split_root.resolve(strict=True)):
                    raise ValueError(f"E8 {split}: shard path escapes its cache directory")
                if shard.get("sample_identity_sha256") != sample_identity["sample_identity_sha256"]:
                    raise ValueError(f"E8 {split}: shard source identity differs from official row order")
                arrays = load_expert_shard(
                    shard_path, expert_identity=cache_identity,
                    sample_identity=sample_identity, feature_dim=CORAL_FEATURE_DIM,
                )
                if (shard.get("file_sha256") != sha256_file(shard_path)
                        or arrays["scores"].shape != (count, 1)
                        or arrays["features"].shape != (count, CORAL_FEATURE_DIM)):
                    raise ValueError(f"E8 {split}: shard digest/dimensions differ from its manifest")
                expected_start += count
                observed_count += count
            if observed_count != expected_count:
                raise ValueError(f"E8 {split}: verified {observed_count}/{expected_count} rows")
            split_reports[split] = {
                "sample_count": observed_count,
                "cache_manifest": str(manifest_path),
                "cache_manifest_sha256": manifest["manifest_sha256"],
                "validated_shards": len(manifest["shards"]),
                "expert_cache_id": cache_identity["cache_id"],
            }
        finally:
            del dataset
            gc.collect()
    report = {
        "status": "PASS", "expert": "expert8",
        "expert_cache_identity": cache_identity,
        "base_m2d_checkpoint_sha256": E8_BASE_SHA256,
        "lora_checkpoint_sha256": E8_LORA_SHA256,
        "splits": split_reports,
    }
    atomic_write_json(report_path, report)
    return report


if RUN_E8_EXTRACTION:
    run_logged(
        "expert8_extract",
        [sys.executable, REPO_ROOT / "scripts/extract_expert8_features.py",
         "--config", E8_CONFIG, "--dataset-root", DATA_ROOT,
         "--output-root", E8_CACHE_ROOT, "--device", "cuda:0",
         "--batch-size", str(E8_BATCH_SIZE), "--shard-size", str(E8_SHARD_SIZE),
         "--report-json", E8_EXTRACTION_REPORT],
        env=gpu_env(0),
    )
else:
    print("E8 extraction disabled; independently verifying every existing source row and shard.")
e8_extraction = verify_existing_e8_cache(
    E8_CONFIG, DATA_ROOT, E8_CACHE_ROOT, E8_EXTRACTION_REPORT,
)
if (e8_extraction.get("status") != "PASS"
        or e8_extraction["splits"].get("train", {}).get("sample_count") != EXPECTED_COUNTS["train"]
        or e8_extraction["splits"].get("validation", {}).get("sample_count") != EXPECTED_COUNTS["validation"]
        or e8_extraction["splits"].get("test", {}).get("sample_count") != EXPECTED_COUNTS["test"]):
    raise ValueError("E8 verified cache does not cover all 7,500/3,000/3,000 official rows")
E8_CACHE_ID = e8_extraction["expert_cache_identity"]["cache_id"]
print("E8 cache identity:", E8_CACHE_ID, "LoRA SHA-256:", E8_LORA_SHA256)


## 15 — Assemble Full8 and run the train-only Score Audit


In [ ]:
FULL8_BUILD_REPORT=LOG_ROOT/"full8_build.json"
if RUN_FULL8_BUILD:
    command=[sys.executable,REPO_ROOT/"scripts/build_full8_feature_store.py","--full7-root",FULL7_ROOT,"--expert8-cache-root",E8_CACHE_ROOT,"--expert8-cache-id",E8_CACHE_ID,"--output-root",FULL8_ROOT,"--split","train","--split","validation","--split","test","--report-json",FULL8_BUILD_REPORT]
    if ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP: command.append("--allow-train-validation-audio-overlap")
    run_logged("full8_build",command)
full8_manifests={}
for split,expected in EXPECTED_COUNTS.items():
    manifest=validate_feature_store(FULL8_ROOT,split=split,require_labels=split!="test")
    count=sum(shard["count"] for shard in manifest["shards"])
    if (manifest.get("expert_profile"),manifest.get("feature_width"),manifest.get("score_width"),count)!=("full_8",33032,8,expected): raise ValueError(f"{split}: Full8 profile/dimension/count mismatch")
    if manifest["labels_present"] is (split=="test"): raise ValueError(f"{split}: Full8 test store must not contain labels")
    full8_manifests[split]=manifest; print(split,"Full8 verified",count,manifest["store_identity_sha256"])
if RUN_SCORE_AUDIT:
    FULL8_AUDIT_PATH=LOG_ROOT/"full8_score_audit.json"
    run_logged("full8_train_only_score_audit",[sys.executable,REPO_ROOT/"scripts/audit_full7_scores.py","--store-root",FULL8_ROOT,"--output-json",FULL8_AUDIT_PATH])
    full8_audit=json.loads(FULL8_AUDIT_PATH.read_text())
    if full8_audit.get("status")!="PASS" or set(full8_audit["diagnostics"]["experts"])!=set(full8_manifests["train"]["expert_order"]): raise ValueError("Full8 train-only Score Audit failed or omitted an expert")


## 16 — Configure Full8 Gate and select a compatible checkpoint run

A completed run is reused only when manifest, latest and best checkpoints match current source, configuration, train/validation stores, all eight expert identities, review policy and overlap diagnostic. Incomplete runs resume only when both checkpoints are compatible. Unsafe artifacts remain preserved in their old directory; fresh runs have no `--resume`.


In [ ]:
from training.config import load_moe_training_config
from training.checkpoint_manager import FULL8_CHECKPOINT_FORMAT_VERSION,FULL8_GATE_ARCHITECTURE,full8_source_code_sha256,_fingerprint,MoeCheckpointManager
from training.full8_resume_policy import choose_full8_gate_run,gate_resume_args
from data.moe_training_dataset import MoeTrainingDataset
from scripts.train_moe_gating import _checkpoint_config,_store_provenance,_real_score_review
from model.moe_gating_core import GatingNetwork
from training.moe_trainer import MoeGateTrainer
GATE_CONFIG=CONFIG_ROOT/"moe_full8.yaml"
GATE_TEMPLATE=yaml.safe_load((REPO_ROOT/"configs/moe_full8.example.yaml").read_text())
GATE_TEMPLATE["data"].update({"feature_store_root":str(FULL8_ROOT),"profile":"full_8","allow_train_validation_audio_overlap":ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP})
GATE_TEMPLATE["runtime"]["device"]=GATE_DEVICE
if SCORE_REVIEW_FILE_OVERRIDE is not None: GATE_TEMPLATE["data"]["score_review_file"]=str(Path(SCORE_REVIEW_FILE_OVERRIDE).expanduser().resolve())
GATE_TEMPLATE["output_dir"]=str(GATE_ROOT)
def write_gate_config(): GATE_CONFIG.write_text(yaml.safe_dump(GATE_TEMPLATE,sort_keys=False))
train_gate_ds=MoeTrainingDataset(str(FULL8_ROOT),split="train",require_labels=True)
validation_gate_ds=MoeTrainingDataset(str(FULL8_ROOT),split="validation",require_labels=True)
expert_checkpoint_provenance,GATE_STORE_IDENTITY=_store_provenance(train_gate_ds,validation_gate_ds,allow_train_validation_audio_overlap=ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP)
GATE_TRAIN_STORE_ID = train_gate_ds.identity
GATE_VALIDATION_STORE_ID = validation_gate_ds.identity
synthetic_provenance=bool(train_gate_ds.manifest["assembly_identity"].get("synthetic_provenance"))
REVIEW_POLICY="SYNTHETIC_NOT_APPLICABLE" if synthetic_provenance else ("EXPERIMENTAL_SKIPPED" if SKIP_SCORE_REVIEW_EXPERIMENTAL else "HUMAN_REVIEWED")
GATE_STORE_IDENTITY["score_review_policy"]=REVIEW_POLICY
if REVIEW_POLICY == "HUMAN_REVIEWED":
    if not RUN_SCORE_AUDIT or "full8_audit" not in globals():
        raise ValueError("Strict Full8 mode requires the completed train-only Score Audit")
    GATE_STORE_IDENTITY["score_review_sha256"] = _real_score_review(
        SCORE_REVIEW_FILE_OVERRIDE, train_gate_ds.manifest, full8_audit,
    )
if not SKIP_SCORE_REVIEW_EXPERIMENTAL and not synthetic_provenance and SCORE_REVIEW_FILE_OVERRIDE is None: raise ValueError("Strict mode requires a genuine review file in SCORE_REVIEW_FILE_OVERRIDE")
del train_gate_ds, validation_gate_ds
gc.collect()
write_gate_config(); loaded_config=load_moe_training_config(GATE_CONFIG); loaded_config["data"]["score_review_policy"]=REVIEW_POLICY
EXPECTED_RUN_CONFIG=_checkpoint_config(loaded_config)
EXPECTED_CONFIG_SHA256=hashlib.sha256(json.dumps(EXPECTED_RUN_CONFIG,sort_keys=True,separators=(",",":"),allow_nan=False).encode()).hexdigest()
EXPECTED_SOURCE_CODE_SHA256=full8_source_code_sha256(); EXPECTED_OVERLAP_DIAGNOSTIC=GATE_STORE_IDENTITY["audio_overlap_diagnostic"]

def compatible_gate_checkpoint(path):
    path=Path(path)
    if not path.is_file(): return False
    try:
        probe=GatingNetwork(33032,8,profile="full_8")
        trainer_probe=MoeGateTrainer(
            probe,EXPECTED_RUN_CONFIG["training"]["loss"],
            learning_rate=EXPECTED_RUN_CONFIG["training"]["learning_rate"],
            weight_decay=EXPECTED_RUN_CONFIG["training"]["weight_decay"],
            diversity_weight=EXPECTED_RUN_CONFIG["training"]["diversity_weight"],
        )
        payload=MoeCheckpointManager().load(
            path,gating_network=probe,optimizer=trainer_probe.optimizer,
            scheduler=trainer_probe.scheduler,expected_config=EXPECTED_RUN_CONFIG,
            expected_feature_store_identity=GATE_STORE_IDENTITY,
            expected_expert_checkpoint_provenance=expert_checkpoint_provenance,
            restore_rng=False,
        )
        del trainer_probe,probe
        return (payload.get("format_version")==FULL8_CHECKPOINT_FORMAT_VERSION
            and payload.get("gate_architecture")==FULL8_GATE_ARCHITECTURE
            and payload.get("source_code_sha256")==EXPECTED_SOURCE_CODE_SHA256
            and payload.get("config")==EXPECTED_RUN_CONFIG
            and payload.get("config_sha256")==_fingerprint(payload.get("config"),"config")==EXPECTED_CONFIG_SHA256
            and payload.get("feature_store_identity")==GATE_STORE_IDENTITY
            and payload.get("feature_store_identity_sha256")==_fingerprint(payload.get("feature_store_identity"),"feature_store_identity")
            and payload.get("expert_checkpoint_provenance")==expert_checkpoint_provenance
            and payload.get("expert_checkpoint_provenance_sha256")==_fingerprint(payload.get("expert_checkpoint_provenance"),"expert_checkpoint_provenance")
            and payload.get("feature_store_identity",{}).get("score_review_policy")==REVIEW_POLICY
            and payload.get("feature_store_identity",{}).get("audio_overlap_diagnostic")==EXPECTED_OVERLAP_DIAGNOSTIC
            and payload.get("selection_policy")=="official_validation_srcc_strict_v1_provisional")
    except Exception as exc:
        print("Gate checkpoint incompatible:",path,f"{type(exc).__name__}: {exc}"); return False

def file_sha(path): return sha256_file(path) if Path(path).is_file() else None
def completed_run_compatible(path,latest_ok,best_ok):
    if not latest_ok or not best_ok or not Path(path).is_file(): return False
    try:
        body=json.loads(Path(path).read_text())
        return (body.get("expert_profile")=="full_8" and body.get("source_code_sha256")==EXPECTED_SOURCE_CODE_SHA256
            and body.get("config_sha256")==EXPECTED_CONFIG_SHA256
            and body.get("train_store_identity_sha256")==GATE_TRAIN_STORE_ID
            and body.get("validation_store_identity_sha256")==GATE_VALIDATION_STORE_ID
            and body.get("expert_checkpoint_provenance")==expert_checkpoint_provenance
            and body.get("score_review_policy")==REVIEW_POLICY
            and body.get("audio_overlap_diagnostic")==EXPECTED_OVERLAP_DIAGNOSTIC
            and body.get("allow_train_validation_audio_overlap")==ALLOW_TRAIN_VALIDATION_AUDIO_OVERLAP
            and body.get("gate_checkpoint_sha256")==file_sha(GATE_ROOT/"best_gate.pt")
            and body.get("latest_gate_checkpoint_sha256")==file_sha(GATE_ROOT/"latest_gate.pt"))
    except Exception as exc: print("Gate manifest incompatible:",path,f"{type(exc).__name__}: {exc}"); return False

latest,best,completed=GATE_ROOT/"latest_gate.pt",GATE_ROOT/"best_gate.pt",GATE_ROOT/"run_manifest.json"
latest_compatible=compatible_gate_checkpoint(latest); best_compatible=compatible_gate_checkpoint(best)
completed_compatible=completed_run_compatible(completed,latest_compatible,best_compatible)
has_artifacts=latest.exists() or best.exists() or completed.exists()
safe_resume_pair=latest_compatible and best_compatible and not completed.exists()
isolated_root=None
if has_artifacts and not completed_compatible and not safe_resume_pair: isolated_root=GATE_ROOT_BASE/f"{GATE_ROOT_BASE.name}_fresh_{time.time_ns()}"
selection=choose_full8_gate_run(GATE_ROOT,latest_path=latest,best_path=best,manifest_path=completed,latest_compatible=latest_compatible,best_compatible=best_compatible,completed_compatible=completed_compatible,isolated_output_dir=isolated_root)
if selection.action=="fresh_isolated":
    previous_root=GATE_ROOT; GATE_ROOT=selection.output_dir; GATE_TEMPLATE["output_dir"]=str(GATE_ROOT); write_gate_config()
    latest,best,completed=GATE_ROOT/"latest_gate.pt",GATE_ROOT/"best_gate.pt",GATE_ROOT/"run_manifest.json"
    resume_checkpoint=None
    print("Incompatible Full8 Gate artifacts preserved at",previous_root,"; starting fresh Full8 Gate training at",GATE_ROOT)
else: resume_checkpoint=selection.resume_checkpoint
print("Full8 Gate action:",selection.action,"output:",GATE_ROOT,"resume:",resume_checkpoint)
if SKIP_SCORE_REVIEW_EXPERIMENTAL and not synthetic_provenance:
    print("\n*** EXPERIMENTAL: HUMAN SCORE REVIEW SKIPPED; train-only Score Audit remains active. No human review is claimed. Research-only. ***\n")
if RUN_GATE_TRAIN and selection.action!="reuse_completed":
    gate_env=gpu_env(0)
    if SKIP_SCORE_REVIEW_EXPERIMENTAL: gate_env["MOESCORE_EXPERIMENTAL_SKIP_SCORE_REVIEW"]="1"
    else: gate_env.pop("MOESCORE_EXPERIMENTAL_SKIP_SCORE_REVIEW",None)
    command=[sys.executable,REPO_ROOT/"scripts/train_moe_gating.py","--config",GATE_CONFIG,"--report-json",LOG_ROOT/"full8_gate_training.json"]
    command+=gate_resume_args(selection)
    run_logged("full8_gate_train",command,env=gate_env)
elif selection.action=="reuse_completed": print("Reusing compatible completed Full8 run; no retraining.")
else: print("RUN_GATE_TRAIN=False; requiring a compatible existing best checkpoint.")
if not best.is_file() or not compatible_gate_checkpoint(best): raise FileNotFoundError("Compatible Full8 best_gate.pt required before evaluation and inference")
if selection.action in {"fresh","fresh_isolated"} and resume_checkpoint is not None: raise AssertionError("Fresh output directory retained a stale resume path")


## 17 — Evaluate the official validation split

Validation runs without optimizer updates. If train/validation WAV hashes overlap, the report must include the potentially optimistic-bias warning.


In [ ]:
if RUN_VALIDATION:
    VALIDATION_REPORT=LOG_ROOT/"full8_validation.json"
    run_logged("full8_validation",[sys.executable,REPO_ROOT/"scripts/evaluate_moe_gating.py","--config",GATE_CONFIG,"--checkpoint",best,"--device","cuda:0","--report-json",VALIDATION_REPORT])
    validation_report=json.loads(VALIDATION_REPORT.read_text())
    if validation_report.get("status")!="PASS" or validation_report.get("optimizer_steps")!=0: raise ValueError("Full8 validation failed or used optimizer updates")
    if EXPECTED_OVERLAP_DIAGNOSTIC.get("overlapping_sha256_count") and not validation_report.get("validation_metrics_warning"): raise ValueError("Overlapping validation metrics must be marked potentially biased")


## 18 — Run label-free test inference, freeze predictions, then calculate research metrics

The test labels are opened only in this stage, after a successful label-free inference and SHA-256 freeze. Official labels use `wav_file_name,text,average_score`.


In [ ]:
SUBMISSION=GATE_ROOT/"submission.csv"; FROZEN_SUBMISSION=GATE_ROOT/"submission_frozen.csv"
TEST_FREEZE_MANIFEST=LOG_ROOT/"full8_test_prediction_freeze.json"; test_inference_completed=False
if RUN_TEST_INFERENCE:
    TEST_INFERENCE_REPORT=LOG_ROOT/"full8_test_inference.json"
    run_logged("full8_test_inference",[sys.executable,REPO_ROOT/"scripts/infer_moe_gating.py","--config",GATE_CONFIG,"--checkpoint",best,"--output",SUBMISSION,"--device","cuda:0","--report-json",TEST_INFERENCE_REPORT],env=gpu_env(0))
    inference_report=json.loads(TEST_INFERENCE_REPORT.read_text())
    if inference_report.get("status")!="PASS" or inference_report.get("labels_required") is not False or inference_report.get("sample_count")!=EXPECTED_COUNTS["test"]: raise ValueError("Full8 label-free inference failed or did not cover all test rows")
    with SUBMISSION.open(encoding="utf-8",newline="") as stream:
        reader=csv.DictReader(stream)
        if reader.fieldnames!=["wave_file_name","pred_score"]: raise ValueError(f"Unexpected submission schema: {reader.fieldnames}")
        predictions=list(reader)
    if len(predictions)!=EXPECTED_COUNTS["test"]: raise ValueError("Submission row count mismatch")
    prediction_sha=sha256_file(SUBMISSION); shutil.copy2(SUBMISSION,FROZEN_SUBMISSION); FROZEN_SUBMISSION.chmod(0o444)
    if sha256_file(FROZEN_SUBMISSION)!=prediction_sha: raise ValueError("Frozen prediction digest mismatch")
    atomic_write_json(TEST_FREEZE_MANIFEST,{"status":"FROZEN_AFTER_LABEL_FREE_INFERENCE","sample_count":EXPECTED_COUNTS["test"],"prediction_file":str(FROZEN_SUBMISSION),"prediction_sha256":prediction_sha,"gate_checkpoint_sha256":sha256_file(best)})
    test_inference_completed=True; print("Frozen label-free test predictions:",FROZEN_SUBMISSION,prediction_sha)
else: print("Test inference skipped; test labels will not be read.")


In [ ]:
TEST_METRICS_PATH=LOG_ROOT/"test_metrics.json"
if not RUN_TEST_METRICS:
    test_metrics_report={"status":"SKIPPED","run_purpose":"research_only","reason":"RUN_TEST_METRICS=False; test labels were not accessed."}; atomic_write_json(TEST_METRICS_PATH,test_metrics_report)
elif not test_inference_completed:
    test_metrics_report={"status":"SKIPPED","run_purpose":"research_only","reason":"No inference/frozen predictions in this run; test labels were not accessed."}; atomic_write_json(TEST_METRICS_PATH,test_metrics_report)
elif not ALLOW_POST_INFERENCE_TEST_LABEL_ACCESS:
    test_metrics_report={"status":"SKIPPED_LABEL_ACCESS_DISABLED","run_purpose":"research_only","reason":"Post-inference test-label access is disabled."}; atomic_write_json(TEST_METRICS_PATH,test_metrics_report)
else:
    # First test-label read in this notebook is after frozen predictions.
    TEST_LABELS=DATA_ROOT/"meta_data"/"test_average.csv"
    if not TEST_LABELS.is_file():
        test_metrics_report={"status":"SKIPPED_LABELS_UNAVAILABLE","run_purpose":"research_only","reason":"Official labels unavailable; no metrics fabricated."}; atomic_write_json(TEST_METRICS_PATH,test_metrics_report)
    else:
        frozen=json.loads(TEST_FREEZE_MANIFEST.read_text())
        if sha256_file(FROZEN_SUBMISSION)!=frozen["prediction_sha256"]: raise ValueError("Frozen predictions changed before metrics")
        from scripts.full7_test_metrics import evaluate_post_inference_metrics
        test_metrics_report=evaluate_post_inference_metrics(FROZEN_SUBMISSION,TEST_LABELS,TEST_METRICS_PATH,expected_count=EXPECTED_COUNTS["test"])
        test_metrics_report["submission_file"]=str(SUBMISSION); test_metrics_report["submission_sha256"]=sha256_file(SUBMISSION)
        if test_metrics_report["prediction_sha256"]!=test_metrics_report["submission_sha256"]: raise ValueError("Metrics were not calculated from the final submission")
        atomic_write_json(TEST_METRICS_PATH,test_metrics_report)
print("Post-inference metrics:",test_metrics_report)


## 19 — Save checkpoints, reports, and local backup artifacts


In [ ]:
if CREATE_LOCAL_BACKUP:
    archive=WORK_ROOT/"moescore_full8_artifacts.tar.gz"
    artifacts=[(best,"gate/best_gate.pt"),(latest,"gate/latest_gate.pt"),(completed,"gate/run_manifest.json"),(SUBMISSION,"submission.csv"),(FROZEN_SUBMISSION,"submission_frozen.csv"),(TEST_FREEZE_MANIFEST,"test_prediction_freeze.json"),(GATE_CONFIG,"configs/moe_full8.yaml"),(E8_CONFIG,"configs/expert8_extraction.yaml"),(E8_EXTRACTION_REPORT,"reports/expert8_extraction.json"),(FULL8_BUILD_REPORT,"reports/full8_build.json"),(LOG_ROOT/"full8_score_audit.json","reports/full8_score_audit.json"),(LOG_ROOT/"full8_validation.json","reports/full8_validation.json"),(LOG_ROOT/"full8_gate_training.json","reports/full8_gate_training.json"),(TEST_METRICS_PATH,"reports/test_metrics.json"),(LOG_ROOT/"full8_test_inference.json","reports/test_inference.json")]
    if "whisper_archive_report" in globals():
        artifacts.extend([(WHISPER_ARCHIVE,"whisper/whisper_feature_cache.tar"),
                          (WHISPER_ARCHIVE_MANIFEST,"whisper/archive_manifest.json")])
    with tarfile.open(archive,"w:gz") as tar:
        for artifact,member in artifacts:
            if Path(artifact).is_file(): tar.add(artifact,arcname=member,recursive=False)
        for log_path in sorted(LOG_ROOT.glob("*.log")): tar.add(log_path,arcname=str(Path("logs")/log_path.name),recursive=False)
    print("Full8 archive:",archive,"bytes:",archive.stat().st_size)
print("Full8 store/Gate/submission:",FULL8_ROOT,GATE_ROOT,SUBMISSION)
